<a href="https://colab.research.google.com/github/NinhHTK/Documents/blob/main/SE196288_Design_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'tiny-imagenet' dataset.
Path to dataset files: /kaggle/input/tiny-imagenet


In [15]:
import os
import kagglehub

try:
    base_path = path
except NameError:
    base_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")

# Chuẩn hóa đường dẫn để tránh bị lồng thư mục
while True:
    subdirs = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]
    if "tiny-imagenet-200" in subdirs:
        base_path = os.path.join(base_path, "tiny-imagenet-200")
    elif len(subdirs) == 1 and subdirs[0] == os.path.basename(base_path):
        base_path = os.path.join(base_path, subdirs[0])
    else:
        break

print(f"Dataset path: {base_path}")

def list_dir_structure(path_to_list, max_depth=2, current_depth=0):
    if current_depth > max_depth:
        return
    try:
        items = os.listdir(path_to_list)
    except Exception as e:
        print(f"Error: {e}")
        return

    indent = "  " * current_depth
    for item in sorted(items)[:10]:
        item_path = os.path.join(path_to_list, item)
        if os.path.isdir(item_path):
            print(f"{indent}[DIR] {item}")
            list_dir_structure(item_path, max_depth, current_depth + 1)
        else:
            print(f"{indent}[FILE] {item}")
    if len(items) > 10:
        print(f"{indent}... and {len(items) - 10} more items")

list_dir_structure(base_path)

Dataset path: /kaggle/input/tiny-imagenet/tiny-imagenet-200/tiny-imagenet-200
[DIR] test
  [DIR] images
    [FILE] test_0.JPEG
    [FILE] test_1.JPEG
    [FILE] test_10.JPEG
    [FILE] test_100.JPEG
    [FILE] test_1000.JPEG
    [FILE] test_1001.JPEG
    [FILE] test_1002.JPEG
    [FILE] test_1003.JPEG
    [FILE] test_1004.JPEG
    [FILE] test_1005.JPEG
    ... and 9990 more items
[DIR] train
  [DIR] n01443537
    [DIR] images
    [FILE] n01443537_boxes.txt
  [DIR] n01629819
    [DIR] images
    [FILE] n01629819_boxes.txt
  [DIR] n01641577
    [DIR] images
    [FILE] n01641577_boxes.txt
  [DIR] n01644900
    [DIR] images
    [FILE] n01644900_boxes.txt
  [DIR] n01698640
    [DIR] images
    [FILE] n01698640_boxes.txt
  [DIR] n01742172
    [DIR] images
    [FILE] n01742172_boxes.txt
  [DIR] n01768244
    [DIR] images
    [FILE] n01768244_boxes.txt
  [DIR] n01770393
    [DIR] images
    [FILE] n01770393_boxes.txt
  [DIR] n01774384
    [DIR] images
    [FILE] n01774384_boxes.txt
  [DIR] n01

In [16]:
import numpy as np
import tensorflow as tf
import os

# Configure paths
dataset_dir = os.path.join(path, "tiny-imagenet-200")
train_dir = os.path.join(dataset_dir, "train")
val_dir = os.path.join(dataset_dir, "val")

# Get class names in order
class_names = sorted(os.listdir(train_dir))
class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}

# Helper to load training image paths and labels
def get_train_samples():
    image_paths = []
    labels = []
    for cls in class_names:
        cls_img_dir = os.path.join(train_dir, cls, "images")
        for img_name in os.listdir(cls_img_dir):
            if img_name.endswith(".JPEG"):
                image_paths.append(os.path.join(cls_img_dir, img_name))
                labels.append(class_to_idx[cls])
    return image_paths, labels

# Helper to load validation image paths and labels from val_annotations.txt
def get_val_samples():
    image_paths = []
    labels = []
    annotations_file = os.path.join(val_dir, "val_annotations.txt")
    with open(annotations_file, "r") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 2:
                img_name, class_name = parts[0], parts[1]
                image_paths.append(os.path.join(val_dir, "images", img_name))
                labels.append(class_to_idx[class_name])
    return image_paths, labels

train_paths, train_labels = get_train_samples()
val_paths, val_labels = get_val_samples()

print(f"Loaded {len(train_paths)} training images.")
print(f"Loaded {len(val_paths)} validation images.")

Loaded 100000 training images.
Loaded 10000 validation images.


In [25]:
# Cấu hình Dataset hiệu năng cao với Augmentation được giảm nhẹ để tăng tốc huấn luyện
IMG_SIZE = 64
BATCH_SIZE = 128

def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = image / 255.0  # Chuẩn hóa về [0, 1]
    return image, label

# Giảm nhẹ Data Augmentation (chỉ giữ RandomFlip để tăng tốc độ tính toán)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal")
])

# Khởi tạo Datasets
train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = train_ds.shuffle(buffer_size=10000)
train_ds = train_ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_ds = val_ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE)
val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

print("Datasets đã sẵn sàng với Augmentation tối giản để tối ưu thời gian!")

Datasets đã sẵn sàng với Augmentation tối giản để tối ưu thời gian!


In [7]:
# Define our custom Convolutional Neural Network (CNN) manually without importing pre-existing network backbones.
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    # Block 1
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),

    # Block 2
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    # Block 3
    layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.4),

    # Classification Head
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(200, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 64, 64, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 16, 16, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 16, 16, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 16, 16, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     8,389,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 9,642,760 (36.78 MB)

 Trainable params: 9,639,944 (36.77 MB)

 Non-trainable params: 2,816 (11.00 KB)

In [26]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

def residual_block(x, filters, kernel_size=3, stride=1):
    shortcut = x

    # Nhánh chính với L2 giảm xuống 1e-5
    x_path = layers.Conv2D(filters, kernel_size, strides=stride, padding='same',
                           kernel_regularizer=regularizers.l2(1e-5))(x)
    x_path = layers.BatchNormalization()(x_path)
    x_path = layers.Activation('relu')(x_path)

    x_path = layers.Conv2D(filters, kernel_size, strides=1, padding='same',
                           kernel_regularizer=regularizers.l2(1e-5))(x_path)
    x_path = layers.BatchNormalization()(x_path)

    # Điều chỉnh kích thước của shortcut nếu số filter hoặc stride thay đổi
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same',
                                 kernel_regularizer=regularizers.l2(1e-5))(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    # Cộng kết nối tắt
    x = layers.add([x_path, shortcut])
    x = layers.Activation('relu')(x)
    return x

# Thiết kế thủ công mô hình ResNet-style CNN
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# Khối Augmentation tích hợp GPU
x = data_augmentation(inputs)

# Tầng Convolution đầu tiên với L2 = 1e-5
x = layers.Conv2D(64, 3, strides=1, padding='same', kernel_regularizer=regularizers.l2(1e-5))(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)

# Các cụm Residual Blocks
x = residual_block(x, 64)
x = residual_block(x, 64)
x = layers.Dropout(0.2)(x)

x = residual_block(x, 128, stride=2)
x = residual_block(x, 128)
x = layers.Dropout(0.3)(x)

x = residual_block(x, 256, stride=2)
x = residual_block(x, 256)
x = layers.Dropout(0.4)(x)

x = residual_block(x, 512, stride=2)
x = residual_block(x, 512)
x = layers.Dropout(0.4)(x)

# Global Average Pooling
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(1e-5))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(200, activation='softmax')(x)

model = models.Model(inputs=inputs, outputs=outputs)

# Thiết lập học bổng ban đầu giảm xuống 0.0001 (1e-4) để tránh quá nhiệt
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

model.summary()

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_9       │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_4        │ (None, 64, 64, 3) │          0 │ input_layer_9[0]… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_64 (Conv2D)  │ (None, 64, 64,    │      1,792 │ sequential_4[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_64[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_44       │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_65 (Conv2D)  │ (None, 64, 64,    │     36,928 │ activation_44[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_65[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_45       │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_66 (Conv2D)  │ (None, 64, 64,    │     36,928 │ activation_45[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_66[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_20 (Add)        │ (None, 64, 64,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │ activation_44[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_46       │ (None, 64, 64,    │          0 │ add_20[0][0]      │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_67 (Conv2D)  │ (None, 64, 64,    │     36,928 │ activation_46[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_67[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_47       │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_68 (Conv2D)  │ (None, 64, 64,    │     36,928 │ activation_47[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_68[0][0] 

 Total params: 11,550,536 (44.06 MB)

 Trainable params: 11,539,912 (44.02 MB)

 Non-trainable params: 10,624 (41.50 KB)

In [27]:
# Huấn luyện mô hình tối ưu với giới hạn thời gian ~1 giờ
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.5,
    patience=2, # Giảm xuống 2 để phản ứng nhanh hơn trong chu kỳ 10 epochs
    min_lr=1e-6,
    verbose=1
)

# Giới hạn số Epochs xuống còn 10 để hoàn thành huấn luyện nhanh chóng
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[lr_scheduler]
)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 398s 447ms/step - accuracy: 0.0933 - loss: 4.8934 - val_accuracy: 0.0414 - val_loss: 6.3765 - learning_rate: 1.0000e-04
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 321s 411ms/step - accuracy: 0.1488 - loss: 4.3422 - val_accuracy: 0.0494 - val_loss: 6.2532 - learning_rate: 1.0000e-04
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 320s 409ms/step - accuracy: 0.1907 - loss: 3.9287 - val_accuracy: 0.0512 - val_loss: 5.6525 - learning_rate: 1.0000e-04
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 320s 410ms/step - accuracy: 0.2217 - loss: 3.6532 - val_accuracy: 0.0717 - val_loss: 5.5024 - learning_rate: 1.0000e-04
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 320s 409ms/step - accuracy: 0.2471 - loss: 3.4454 - val_accuracy: 0.0785 - val_loss: 5.2182 - learning_rate: 1.0000e-04
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 321s 410ms/step - accuracy: 0.2726 - loss: 3.2702 - val_accuracy: 0.0817 - val_loss: 5.1912 - learning_rate: 1.0000e-04
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 

In [28]:
# Re-compile với Learning Rate nhỏ hơn để tinh chỉnh mịn hơn (Fine-tuning)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Chạy nối tiếp 10 Epochs nữa (từ Epoch 11 -> 20)
history_extra = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[lr_scheduler]
)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 363s 433ms/step - accuracy: 0.3658 - loss: 2.7704 - val_accuracy: 0.1234 - val_loss: 4.4102 - learning_rate: 5.0000e-05
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 321s 411ms/step - accuracy: 0.3910 - loss: 2.6331 - val_accuracy: 0.1319 - val_loss: 4.3947 - learning_rate: 5.0000e-05
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 320s 409ms/step - accuracy: 0.4068 - loss: 2.5513 - val_accuracy: 0.1353 - val_loss: 4.3331 - learning_rate: 5.0000e-05
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 321s 410ms/step - accuracy: 0.4201 - loss: 2.4812 - val_accuracy: 0.1392 - val_loss: 4.3631 - learning_rate: 5.0000e-05
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 320s 408ms/step - accuracy: 0.4319 - loss: 2.4292 - val_accuracy: 0.1541 - val_loss: 4.2008 - learning_rate: 5.0000e-05
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 319s 408ms/step - accuracy: 0.4438 - loss: 2.3641 - val_accuracy: 0.1611 - val_loss: 4.1360 - learning_rate: 5.0000e-05
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 

In [30]:
model.save("resnet_tiny_imagenet_epoch2.keras")
print("Đã lưu mô hình an toàn!")

Đã lưu mô hình an toàn!


In [ ]:
model.load_weights("resnet_tiny_imagenet_epoch2.keras") # Hoặc tên file checkpoint bạn mới save